# F7 : Détection d'anomalies sur le S&P 500 (Isolation Forest)

**Cours BD & DS · ENCG Settat · S7** · Examen individuel (Partie B)

| | |
|---|---|
| **Nom et prénom** | OUHAJ Mohammed |
| **Code Apogée** | 23008724 |
| **Groupe** | 3 |
| **Algorithme principal** | n°24, Isolation Forest |
| **Données** | FRED, série SP500 (Federal Reserve Bank of St. Louis) |

**Résumé en trois lignes.** On cherche les jours atypiques du S&P 500 sans étiquette, avec un Isolation Forest, et on les compare à la règle des trois écarts-types sur 47 jours de test. L'Isolation Forest signale plus de jours que la règle 3σ, mais ces jours forment surtout une phase de forte volatilité. Les limites (une seule année, taux de contamination fixé à l'avance, résultat sensible à la graine) sont discutées en section 10.

Lancer **Exécution > Tout exécuter** : le notebook télécharge les données et refait toute l'analyse, dans l'ordre du rapport.

## 0. Configuration

In [ ]:
import platform, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

GRAINE = 42                      # graine aléatoire unique du projet
np.random.seed(GRAINE)

RACINE = Path('.')
DONNEES = RACINE / 'data' / 'raw'
FIGURES = RACINE / 'figures'
for d in (DONNEES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# Période étudiée, fixée à l'avance
PERIODE_DEBUT, PERIODE_FIN = '2025-11-17', '2026-11-16'

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
R = {}                           # résultats chiffrés repris dans le rapport

print('Python', platform.python_version(), '| pandas', pd.__version__,
      '| numpy', np.__version__, '| scikit-learn', sklearn.__version__, '| matplotlib', matplotlib.__version__)

## 1. Introduction

**Contexte.** Les marchés actions connaissent des journées atypiques (chocs de prix, phases de tension). Les repérer tôt intéresse les gestionnaires de risque et les analystes de marché, qui peuvent alors déclencher une surveillance ou revoir une exposition.

**Problématique.** Un algorithme non supervisé, l'Isolation Forest, repère-t-il davantage de jours atypiques du S&P 500 que la règle classique des trois écarts-types, et ces jours forment-ils des chocs isolés ou une phase de marché ?

**Objectif mesurable.** Sur les 47 derniers jours de la période (test), comparer le nombre et les dates des anomalies signalées par l'Isolation Forest (taux de contamination fixé **à l'avance** à 5 %) et par la règle des trois écarts-types (référence naïve, calculée sur l'apprentissage), puis mesurer la stabilité du résultat selon la graine. Aucun seuil de réussite chiffré n'est fixé : sans étiquette de « vraie » anomalie, la justesse n'est pas mesurable. L'objectif d'origine (« détecter 4 événements sur 5 ») est abandonné, car les cinq événements n'étaient ni listés ni vérifiables.

**Nature du problème.** Détection d'anomalies non supervisée : il n'existe aucune étiquette de « jour anormal ».

## 2. État de l'art

La détection d'anomalies recouvre plusieurs familles de méthodes : statistiques, fondées sur la distance ou la densité, et fondées sur l'isolement des points [2]. Les règles statistiques comme celle des trois écarts-types sont simples mais supposent une distribution stable et ne regardent qu'une variable à la fois. L'Isolation Forest [3] isole les points atypiques par des partitions aléatoires, sans modéliser la densité des points normaux.

| Approche | Principe | Atouts | Limites | Réf. |
| --- | --- | --- | --- | --- |
| Règle des trois écarts-types (référence naïve) | Signaler une valeur à plus de trois écarts-types de la moyenne | Simple, transparente | Une seule variable ; suppose une loi stable | [2] |
| Approches statistiques classiques | Modéliser la loi des données et signaler les valeurs peu probables | Interprétables | Hypothèses de loi parfois irréalistes | [2] |
| Isolation Forest | Isoler les points par des coupures aléatoires ; une anomalie s'isole en peu de coupures | Multivarié, rapide, sans hypothèse de loi | Résultat sensible à la graine et au taux de contamination | [3] |

## 3. Travaux liés

| Travail | Données | Méthode | Résultat principal | Différence avec cette étude |
| --- | --- | --- | --- | --- |
| **À COMPLÉTER** (auteurs, année) | | | | |
| **À COMPLÉTER** (auteurs, année) | | | | |

**Positionnement.** **À COMPLÉTER** après avoir trouvé et lu deux études réelles proches du sujet (même question, mêmes données ou même méthode). Ne citer que des sources réellement consultées.

## 4. Jeu de données

### 4.1 Téléchargement depuis FRED

In [ ]:
import urllib.request

FICHIER = DONNEES / 'sp500.csv'
URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500'

if not FICHIER.exists():
    try:
        urllib.request.urlretrieve(URL, FICHIER)
        print('Téléchargé depuis FRED le', datetime.date.today().isoformat())
    except Exception as e:
        raise RuntimeError(
            "Téléchargement impossible. Téléchargez le CSV à l'adresse " + URL +
            " et placez-le dans data/raw/sp500.csv, puis relancez.") from e
else:
    print('Fichier déjà présent :', FICHIER)

In [ ]:
brut = pd.read_csv(FICHIER, na_values=['.'], parse_dates=['observation_date'])
brut = brut.rename(columns={'observation_date': 'DATE'})
# Période fixée à l'avance, identique à celle du rapport
df_raw = brut[(brut['DATE'] >= PERIODE_DEBUT) & (brut['DATE'] <= PERIODE_FIN)].reset_index(drop=True)
R['n_brut'] = int(len(df_raw))
print('Dimensions du jeu brut :', df_raw.shape)
print('Période :', df_raw['DATE'].min().date(), 'au', df_raw['DATE'].max().date())
df_raw.head()

In [ ]:
df_raw.dtypes.to_frame('type').T

### 4.2 Description

- **Nom et source :** série SP500 (niveau de clôture quotidien du S&P 500), Federal Reserve Bank of St. Louis (FRED) [1], https://fred.stlouisfed.org/series/SP500
- **Date d'extraction :** **À COMPLÉTER** (la date de téléchargement est affichée par la cellule ci-dessus ; si le fichier existait déjà, mettre la date où vous l'avez téléchargé)
- **Licence ou conditions d'utilisation :** non précisée par la source
- **Taille :** voir la sortie ci-dessus (nombre de lignes brutes, période) ; fréquence quotidienne (jours de bourse)
- **Unité d'observation :** un jour de bourse (cours de clôture de l'indice)
- **Variable cible :** aucune (non supervisé)
- **Adéquation :** la série quotidienne permet de calculer rendements et volatilité, donc de repérer des jours atypiques
- **Limites :** valeurs manquantes les jours de fermeture (notées « . » par FRED), une seule année couverte, aucune étiquette d'anomalie

| Variable | Description | Type | Unité | Rôle |
| --- | --- | --- | --- | --- |
| DATE | Date d'observation | date | jour | date |
| SP500 | Niveau de clôture de l'indice | réel | points d'indice | source |
| Log_Return | $r_t = \ln(P_t / P_{t-1})$ | réel | sans unité | explicative |
| Volatility_20d | écart-type mobile de $r_t$ sur 20 jours de bourse | réel | sans unité | explicative |
| Cum_Return_5d | $\ln(P_t / P_{t-5})$ | réel | sans unité | explicative |

## 5. Exploration

### 5.1 Qualité des données

In [ ]:
R['nan'] = int(df_raw['SP500'].isna().sum())
R['doublons_dates'] = int(df_raw['DATE'].duplicated().sum())
print(f"Valeurs manquantes : {R['nan']} sur {len(df_raw)} lignes ({100*R['nan']/len(df_raw):.1f} %)")
print('Dates en double   :', R['doublons_dates'])
print('Dates croissantes :', bool(df_raw['DATE'].is_monotonic_increasing))
df_raw[df_raw['SP500'].isna()]

**Lecture.** Les valeurs manquantes correspondent aux jours où la bourse américaine est fermée (jours fériés). Elles sont **supprimées** plutôt qu'imputées : interpoler créerait des variations de prix fictives pour des jours sans transaction et fausserait la volatilité. Aucune valeur extrême n'est supprimée : en finance un mouvement extrême est souvent réel, et c'est précisément le signal recherché.

In [ ]:
df = df_raw.dropna(subset=['SP500']).copy().sort_values('DATE').reset_index(drop=True)
df['Log_Return'] = np.log(df['SP500'] / df['SP500'].shift(1))
df['Volatility_20d'] = df['Log_Return'].rolling(20).std()
df['Cum_Return_5d'] = np.log(df['SP500'] / df['SP500'].shift(5))
df = df.dropna().reset_index(drop=True)          # lignes sans historique suffisant
R['n_final'] = int(len(df))
print('Dimensions après préparation :', df.shape)
df[['DATE', 'SP500', 'Log_Return', 'Volatility_20d', 'Cum_Return_5d']].head(10)

### 5.2 Statistiques descriptives

In [ ]:
df[['SP500', 'Log_Return', 'Volatility_20d', 'Cum_Return_5d']].describe().T.round(6)

### 5.3 Graphiques

In [ ]:
# Figure 1 : niveau de l'indice
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(df['DATE'], df['SP500'], color='navy', label='S&P 500 (clôture)')
ax.set_title("Évolution du niveau de clôture du S&P 500")
ax.set_xlabel("Date"); ax.set_ylabel("Niveau de l'indice (points)"); ax.legend(loc='upper left')
fig.tight_layout(); fig.savefig(FIGURES / 'fig1_prix.png', dpi=200); plt.show()

**Lecture de la figure 1.** *(À rédiger d'après le graphique : tendance générale, creux et sommets, en citant les niveaux lus sur le graphique.)*

In [ ]:
# Figure 2 : rendements logarithmiques quotidiens
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(df['DATE'], df['Log_Return'], color='darkorange', lw=1, label='Rendement logarithmique')
ax.axhline(0, color='grey', lw=0.8)
ax.set_title("Rendements logarithmiques quotidiens du S&P 500")
ax.set_xlabel("Date"); ax.set_ylabel("Rendement logarithmique (sans unité, par jour)"); ax.legend(loc='upper left')
fig.tight_layout(); fig.savefig(FIGURES / 'fig2_rendements.png', dpi=200); plt.show()
i_min = df['Log_Return'].idxmin()
print('Rendement le plus bas :', df.loc[i_min, 'DATE'].date(), round(df.loc[i_min, 'Log_Return'], 6))

**Lecture de la figure 2.** *(À rédiger : rendements centrés sur zéro, jour du plus fort mouvement négatif indiqué par la sortie ci-dessus.)*

In [ ]:
# Figure 3 : volatilité sur 20 jours
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(df['DATE'], df['Volatility_20d'], color='crimson', label='Volatilité sur 20 jours')
ax.set_title("Volatilité glissante des rendements sur 20 jours de bourse")
ax.set_xlabel("Date"); ax.set_ylabel("Écart-type des rendements (sans unité)"); ax.legend(loc='upper left')
fig.tight_layout(); fig.savefig(FIGURES / 'fig3_volatilite.png', dpi=200); plt.show()
print('Volatilité 20 j, 5 premières valeurs :', df['Volatility_20d'].head().round(4).tolist())
print('Volatilité 20 j, 5 dernières valeurs :', df['Volatility_20d'].tail().round(4).tolist())

**Lecture de la figure 3.** *(À rédiger : comparer la volatilité en début et en fin de période d'après les deux listes ci-dessus.)*

## 6. Préparation

In [ ]:
FEATURES = ['Log_Return', 'Volatility_20d', 'Cum_Return_5d']
X = df[FEATURES]

# Découpage chronologique 80 % / 20 %, sans mélange
coupure = int(len(X) * 0.8)
X_train, X_test = X.iloc[:coupure], X.iloc[coupure:]
d_train, d_test = df['DATE'].iloc[:coupure], df['DATE'].iloc[coupure:]
R['n_train'], R['n_test'] = int(len(X_train)), int(len(X_test))

print(f"Apprentissage : du {d_train.min().date()} au {d_train.max().date()} ({len(X_train)} observations)")
print(f"Test          : du {d_test.min().date()} au {d_test.max().date()} ({len(X_test)} observations)")
assert d_train.max() < d_test.min()               # contrôle : aucun chevauchement
df.iloc[coupure - 2:coupure + 2][['DATE'] + FEATURES]   # contrôle de l'alignement des dates à la coupure

**Choix de préparation.**

1. **Nettoyage :** suppression des jours sans cours ; valeurs extrêmes conservées.
2. **Variables créées :** rendement logarithmique, volatilité sur 20 jours, rendement cumulé sur 5 jours, toutes calculées avec le passé seulement.
3. **Mise à l'échelle :** `StandardScaler` dans un `Pipeline`, ajusté sur l'apprentissage seulement. Les arbres sont insensibles à l'échelle ; la standardisation est conservée pour la lisibilité et ne change pas le principe.
4. **Découpage chronologique :** les 80 % de jours les plus anciens pour l'apprentissage, les 20 % les plus récents pour le test. L'affichage ci-dessus confirme que le test commence juste après l'apprentissage.

**Prévention des fuites.**

- ☑ Transformations ajustées sur l'apprentissage seulement, dans un `Pipeline`.
- ☑ Découpage chronologique, sans mélange.
- ☑ Variables construites avec le passé seulement.
- ▫ Aucune variable ne contient la cible : sans objet (non supervisé).
- ☑ Jeu de test utilisé une seule fois pour l'évaluation finale : le taux de contamination est fixé à l'avance (5 %) ; la grille de la section 9 est une analyse de sensibilité, pas une sélection du modèle.

## 7. Algorithme

**Choix.** Algorithme n°24 de l'annexe A de l'énoncé : Isolation Forest [3]. Il convient à un problème sans étiquette, accepte plusieurs variables à la fois et n'exige pas de loi de probabilité. La règle des trois écarts-types sert de référence, mais elle ne regarde que le rendement du jour.

**Principe.** L'algorithme construit des arbres dont chaque coupure choisit une variable et un seuil au hasard. Un point atypique est séparé des autres en peu de coupures (chemin court dans l'arbre) ; un point ordinaire en demande davantage. Le score d'anomalie combine les longueurs de chemin sur l'ensemble des arbres. Implémentation : `IsolationForest` de scikit-learn [4].

**Formule essentielle.**

$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$

où $h(x)$ est la longueur du chemin de $x$ dans un arbre, $E(h(x))$ sa moyenne sur les arbres, $n$ le nombre d'observations d'apprentissage et $c(n)$ la longueur moyenne d'un chemin dans un arbre binaire construit sur $n$ points. Un score proche de 1 signale une anomalie. (Dans le code, le score affiché est l'opposé de `decision_function` : plus il est élevé, plus le jour est atypique ; ce n'est pas $s(x,n)$ lui-même.)

**Paramètres appris.** Les coupures (variable et seuil) tirées au hasard à partir de l'apprentissage ; la moyenne et l'écart-type de chaque variable pour la standardisation.

| Hyperparamètre | Rôle et effet attendu | Valeurs étudiées | Valeur retenue |
| --- | --- | --- | --- |
| `contamination` | Part de points classés anomalies : plus elle est élevée, plus il y a d'alertes | 0,01 ; 0,03 ; 0,05 ; 0,10 (sensibilité) | 0,05 (fixée à l'avance) |
| `random_state` | Graine du tirage aléatoire | 42 ; 100 ; 2025 | 42 |
| `n_estimators` | Nombre d'arbres | 100 (fixé) | 100 |

## 8. Protocole expérimental

| Ensemble | Période ou proportion | Observations |
| --- | --- | --- |
| Apprentissage | 80 % les plus anciens | voir section 6 |
| Validation | aucun jeu de validation | 0 |
| Test | 20 % les plus récents | voir section 6 |

- **Référence naïve :** règle des trois écarts-types sur le rendement logarithmique, avec moyenne et écart-type calculés sur l'apprentissage.
- **Validation :** aucune (pas d'étiquettes). Le taux de contamination est fixé à 5 % avant de voir les résultats ; les 12 configurations de la section 9 servent à mesurer la sensibilité.
- **Métrique principale :** nombre d'anomalies signalées sur le test (sans unité). Elle décrit l'activité du détecteur, pas sa justesse.
- **Métriques secondaires :** score d'anomalie de chaque jour signalé ; stabilité du nombre d'alertes selon la graine.
- **Graine aléatoire :** 42.

In [ ]:
# Référence naïve : règle des trois écarts-types (paramètres calculés sur l'apprentissage uniquement)
moy, sig = X_train['Log_Return'].mean(), X_train['Log_Return'].std()
borne_haute, borne_basse = moy + 3 * sig, moy - 3 * sig
masque_3s = (X_test['Log_Return'] > borne_haute) | (X_test['Log_Return'] < borne_basse)
anom_3s = df.loc[X_test.index[masque_3s], ['DATE', 'SP500', 'Log_Return']]
R['n_3s'] = int(len(anom_3s))
print(f"Bornes 3σ : [{borne_basse:.6f} ; {borne_haute:.6f}]")
print(f"[Règle 3σ] anomalies sur le test : {R['n_3s']}")
anom_3s

## 9. Résultats

Le jeu de test est utilisé pour l'évaluation finale ci-dessous.

In [ ]:
def pipeline_if(contamination, seed):
    return Pipeline([('scaler', StandardScaler()),
                     ('model', IsolationForest(n_estimators=100, contamination=contamination, random_state=seed))])

modele = pipeline_if(0.05, GRAINE).fit(X_train)          # ajusté sur l'apprentissage seulement
pred = modele.predict(X_test)                             # -1 : anomalie
score = -modele.decision_function(X_test)                 # plus élevé = plus atypique
test = df.loc[X_test.index, ['DATE', 'SP500', 'Log_Return', 'Volatility_20d']].copy()
test['IF_Score'] = score
anom_if = test[pred == -1]
R['n_if'] = int(len(anom_if))
print(f"[Isolation Forest, contamination 5 %, graine {GRAINE}] anomalies sur le test : {R['n_if']}")
anom_if.round({'Log_Return': 6, 'Volatility_20d': 6, 'IF_Score': 6})

In [ ]:
# Tableau comparatif
comparaison = pd.DataFrame({'Modèle': ['Règle des trois écarts-types (référence)', 'Isolation Forest (0,05, graine 42)'],
                            'Anomalies signalées (test)': [R['n_3s'], R['n_if']]})
comparaison

In [ ]:
# Analyse de sensibilité : 4 contaminations x 3 graines (n'est PAS une sélection du modèle)
lignes = []
for c in [0.01, 0.03, 0.05, 0.10]:
    for s in [42, 100, 2025]:
        n = int((pipeline_if(c, s).fit(X_train).predict(X_test) == -1).sum())
        lignes.append({'contamination': c, 'graine': s, 'anomalies_test': n})
grille = pd.DataFrame(lignes)
R['grille_min'], R['grille_max'] = int(grille['anomalies_test'].min()), int(grille['anomalies_test'].max())
grille.pivot(index='contamination', columns='graine', values='anomalies_test')

In [ ]:
# Figure 4 : scores des jours signalés
a = anom_if.sort_values('DATE')
fig, ax = plt.subplots(figsize=(8, 3.6))
couleurs = ['#c0392b' if d in set(anom_3s['DATE']) else '#2e5e9e' for d in a['DATE']]
ax.bar(a['DATE'].dt.strftime('%d/%m'), a['IF_Score'], color=couleurs)
for i, v in enumerate(a['IF_Score']):
    ax.text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=8)
ax.set_title("Score d'anomalie des jours signalés par l'Isolation Forest (test)")
ax.set_xlabel("Date des jours signalés"); ax.set_ylabel("Score d'anomalie (sans unité)")
ax.text(0.98, 0.92, 'Rouge : aussi signalé par la règle 3σ', transform=ax.transAxes, ha='right', color='#c0392b', fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / 'fig4_scores.png', dpi=200); plt.show()

In [ ]:
# Figure 5 : jours signalés sur la courbe de prix (test)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(test['DATE'], test['SP500'], color='navy', label='S&P 500 (test)')
ax.scatter(anom_if['DATE'], anom_if['SP500'], color='crimson', zorder=3, label='Isolation Forest')
ax.scatter(anom_3s['DATE'], anom_3s['SP500'], marker='x', color='black', s=80, zorder=4, label='Règle 3σ')
ax.set_title("Jours signalés sur la période de test")
ax.set_xlabel("Date"); ax.set_ylabel("Niveau de l'indice (points)"); ax.legend(loc='upper left')
fig.tight_layout(); fig.savefig(FIGURES / 'fig5_signaux_test.png', dpi=200); plt.show()

In [ ]:
# Analyse des erreurs : les jours signalés sont-ils consécutifs ? et la volatilité correspondante
dates_if = anom_if['DATE'].reset_index(drop=True)
jours_ouvres = df['DATE'].reset_index(drop=True)
rang = jours_ouvres.reset_index().set_index('DATE').loc[dates_if, 'index'].to_numpy()
print('Rangs (jours de bourse) des jours signalés :', rang.tolist())
print('Écarts entre jours signalés successifs :', np.diff(rang).tolist())
print(f"Volatilité 20 j sur les jours signalés : {anom_if['Volatility_20d'].min():.4f} à {anom_if['Volatility_20d'].max():.4f}")
print(f"Volatilité 20 j sur l'apprentissage : {X_train['Volatility_20d'].min():.4f} à {X_train['Volatility_20d'].max():.4f}")

**Lecture des résultats.** *(À rédiger d'après les sorties ci-dessus : nombre d'anomalies de la règle 3σ et de l'Isolation Forest ; jours signalés et jour au score le plus élevé ; fourchette du nombre d'alertes à 5 % selon la graine.)*

**Analyse des erreurs.** Sans étiquette, on ne peut compter ni fausses alertes ni oublis. On peut en revanche vérifier si les jours signalés sont rapprochés (écarts de 1 ou 2 jours de bourse dans la sortie ci-dessus) et si la volatilité sur ces jours dépasse nettement celle observée à l'apprentissage : dans ce cas le modèle signale un changement de niveau de volatilité plutôt que des chocs indépendants. *(À confirmer d'après vos sorties.)*

**Exemples de prédictions sur des jours non vus à l'apprentissage.** *(Choisir trois lignes du tableau des anomalies : date, prédiction, rendement réel, et dire si la règle 3σ les signale aussi.)*

## 10. Discussion des résultats

**Réponse à l'objectif.** L'objectif comparatif est atteint si l'Isolation Forest signale un nombre différent de jours que la règle 3σ (voir section 9). L'objectif d'origine, détecter 4 événements de marché sur 5, n'est **pas vérifiable** : les événements ne sont pas listés et la période étudiée est d'un an.

**Interprétation.** L'Isolation Forest utilise la volatilité sur 20 jours et le rendement cumulé en plus du rendement journalier, ce qui lui permet de signaler des jours dont le rendement seul reste sous le seuil 3σ. Plus d'alertes ne veut pas dire plus de justesse : des jours signalés rapprochés forment probablement une seule phase de marché. Aucune causalité n'est établie.

**Surapprentissage et généralisation.** Il n'y a pas de score d'apprentissage comparable en non supervisé. Le résultat dépend de la graine (voir la grille de sensibilité).

**Comparaison avec les travaux liés.** **À COMPLÉTER** quand la section 3 sera remplie avec de vraies études.

**Limites et biais.** (1) Une seule année de données et un test de 47 jours. (2) Le taux de contamination est un choix a priori qui détermine le nombre d'alertes. (3) Aucune explication historique des mouvements n'est vérifiée par une source : le notebook n'en avance pas. (4) Une détection statistique ne prouve pas un krach.

**Pistes d'amélioration.** Choisir la contamination sur une période de validation séparée du test et allonger l'historique à plusieurs années ; lister à l'avance des événements de référence documentés pour mesurer un taux de détection.

## 11. Conclusion

*(À rédiger en 4 à 6 lignes, avec les chiffres de vos sorties : nombre d'anomalies Isolation Forest contre règle 3σ, ce que cela montre, la limite principale et une perspective.)*

## 12. Références et citations

1. Federal Reserve Bank of St. Louis, « S&P 500 (SP500) », FRED, https://fred.stlouisfed.org/series/SP500, consulté le **À COMPLÉTER**.
2. V. Chandola, A. Banerjee, V. Kumar, « Anomaly detection: A survey », *ACM Computing Surveys*, vol. 41, n° 3, article 15, 2009.
3. F. T. Liu, K. M. Ting, Z.-H. Zhou, « Isolation Forest », *Proceedings of the 8th IEEE International Conference on Data Mining (ICDM)*, p. 413-422, 2008.
4. F. Pedregosa *et al.*, « Scikit-learn: Machine Learning in Python », *Journal of Machine Learning Research*, vol. 12, p. 2825-2830, 2011.
5. Support du cours BD & DS, notebook `Tp_ML.ipynb`, ENCG-2026.

## 13. Annexes

### A. Reproduction

Cloner le dépôt, installer les dépendances (`pip install -r requirements.txt`), puis exécuter ce notebook avec **Redémarrer et tout exécuter**. Les données sont téléchargées dans `data/raw/` et les figures enregistrées dans `figures/`.

### B. Déclaration des outils d'IA

| Outil ou ressource | Date | Usage précis | Parties concernées | Vérification effectuée |
| --- | --- | --- | --- | --- |
| Claude (Anthropic) | **À COMPLÉTER** | Aide à la structuration du notebook, des diapositives et du rapport | **À COMPLÉTER** | **À COMPLÉTER : ce que j'ai relu et vérifié moi-même** |

In [ ]:
# Enregistrement des résultats chiffrés et versions utilisées
import json
R['versions'] = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
Path('resultats.json').write_text(json.dumps(R, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(json.dumps(R, ensure_ascii=False, indent=2, default=str))